In [1]:
import torch
import datasets as hf_datasets
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from tqdm import tqdm

from custom_datasets.qm9 import QM9Dataset
from utils.tokenizer import VocabTokenizer

/home/krojas/Documents/Research/Variable-Length-Diffusion/Variable-Length-Diffusion-Toy/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def plot_molecule(symbols, positions):
    """
    Plots a molecule in 3D based on its QM9-style dictionary.
    
    Args:
        molecule_data (dict): Dictionary containing 'atomic_symbols', 'pos', etc.
    """
    # CPK coloring convention (approximate)
    colors = {
        'H': 'white',
        'C': 'grey',
        'N': 'blue',
        'O': 'red',
        'F': 'green',
        'S': 'yellow',
        'Cl': 'green'
    }
    
    # Atom sizes (approximate relative scales)
    sizes = {
        'H': 100,
        'C': 300,
        'N': 300,
        'O': 300,
        'F': 300,
        'S': 400,
        'Cl': 400
    }
    
    atom_colors = [colors.get(s, 'pink') for s in symbols]
    atom_sizes = [sizes.get(s, 200) for s in symbols]
    
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    
    # Plot atoms
    ax.scatter(positions[:, 0], positions[:, 1], positions[:, 2], 
               s=atom_sizes, c=atom_colors, edgecolor='black', alpha=1.0)
    
    # Infer and plot bonds based on distance
    # Covalent radii (Angstroms)
    radii = {
        'H': 0.31,
        'C': 0.76,
        'N': 0.71,
        'O': 0.66,
        'F': 0.57,
        'S': 1.05,
        'Cl': 1.02
    }
    
    num_atoms = len(symbols)
    for i in range(num_atoms):
        for j in range(i + 1, num_atoms):
            dist = np.linalg.norm(positions[i] - positions[j])
            
            # Simple bond threshold: sum of radii + tolerance
            threshold = radii.get(symbols[i], 0.7) + radii.get(symbols[j], 0.7) + 0.3
            
            if dist < threshold:
                ax.plot([positions[i, 0], positions[j, 0]],
                        [positions[i, 1], positions[j, 1]],
                        [positions[i, 2], positions[j, 2]],
                        color='black', linewidth=2)
    
    # Label atoms
    for i, sym in enumerate(symbols):
        ax.text(positions[i, 0], positions[i, 1], positions[i, 2], sym, 
                fontsize=10, ha='center', va='center', zorder=10)

    # title = molecule_data.get('canonical_smiles', molecule_data.get('smiles', 'Molecule'))
    # ax.set_title(f"Molecule Structure: {title}")
    ax.set_xlabel('X (Å)')
    ax.set_ylabel('Y (Å)')
    ax.set_zlabel('Z (Å)')
    
    # Set equal aspect ratio for 3D plot to prevent distortion
    # Matplotlib 3D doesn't have "axis equal" so we fake it by setting limits
    max_range = np.array([positions[:, 0].max()-positions[:, 0].min(), 
                          positions[:, 1].max()-positions[:, 1].min(), 
                          positions[:, 2].max()-positions[:, 2].min()]).max() / 2.0

    mid_x = (positions[:, 0].max()+positions[:, 0].min()) * 0.5
    mid_y = (positions[:, 1].max()+positions[:, 1].min()) * 0.5
    mid_z = (positions[:, 2].max()+positions[:, 2].min()) * 0.5
    
    ax.set_xlim(mid_x - max_range, mid_x + max_range)
    ax.set_ylim(mid_y - max_range, mid_y + max_range)
    ax.set_zlim(mid_z - max_range, mid_z + max_range)
    
    plt.tight_layout()
    plt.show()



In [3]:
tokenizer = VocabTokenizer(vocab={'H', 'C', 'N', 'O', 'F'})
dataset = QM9Dataset(tokenizer)

In [5]:
data = dataset[10001]
# mask = data['mask']
# # plot_molecule(data['x'][mask], data['y'][mask])
data

{'x': tensor([3, 0, 0, 0, 0, 3, 0, 3, 4, 4, 4, 4, 4, 4, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5,
         5, 5, 5, 5, 5, 5]),
 'y': [[-0.0674766373, -0.1227147817, -0.219306265],
  [-0.108277786, 1.0804895233, -0.2010589117],
  [1.1192628828, 1.9589383373, -0.0351013522],
  [1.3914633734, 2.835489088, -1.2674608689],
  [0.4196397685, 4.0079742718, -1.4104435417],
  [-0.5341932561, 4.1698697912, -0.696525307],
  [2.8193352695, 3.3917212562, -1.3058744931],
  [3.2259585682, 4.0935358417, -2.1922791518],
  [-1.0639981883, 1.6323599832, -0.3115255489],
  [0.9574963554, 2.611389592, 0.8320317214],
  [1.9741205869, 1.3060042751, 0.1638781243],
  [1.292668099, 2.2560439165, -2.1987106342],
  [0.6775176539, 4.7095923775, -2.2278362298],
  [3.4560836303, 3.1106899679, -0.4385878813],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0],
  [0, 0, 0]],
 'mask': tensor(